# 🌊 Satellite Correction - whacs Spectrum (Optional)

> **Inputs:**
> - Wave spectrum to be corrected in `.nc` format, located in the `inputs/` folder.
> - Satellite data in `.nc` format, also located in the `inputs/` folder.
> - *Optional:* Wave buoy data for validating the satellite-based correction.
>
> **⚠️NOTE:** Instructions for downloading the satellite data are available at:  
> https://github.com/javitausia/CalValWaves
>
> **Outputs:**
> - Satellite-corrected wave spectrum in `.nc` format, saved in the `inputs/` folder.  
>   This file can be used either for Super-Point construction (if needed) or directly as input for BinWaves.



---

#### Before You Start

Make sure you have:

- Installed the latest version of `bluemath-tk`:  
  ```bash
  pip install bluemath-tk

- **BlueMath Toolkit Repository**:  [https://github.com/GeoOcean/BlueMath_tk.git](https://github.com/GeoOcean/BlueMath_tk.git)

Before continuing, ensure you have **created and activated a Python environment**.

*** Other Required Packages ***
- `wavespectra` 
- `cartophy`
 
---

This Jupyter Notebook is an alternative step to correct whacs or other hindcast data with Satellite information. The corrected spectra can be use: 

1. Create the SuperPoint (Cagigal et al., 2023)
2. As a input to teh BinWaves Reconstrcution Notebook.

---
<details open>


The hindcast significant wave height data can be calibrated using both buoy and satellite significant height of waves as the "good" measure, but the way we prefer to do it is calibrating first with the satellite data and validating with the buoy after the calibration. A description of what is done can be seen in this paper:

João Albuquerque, Jose A. A. Antolínez, Ana Rueda, Fernando J.Méndez, Giovanni Cocoa (November 2018). Directional correction of modeled sea and swell wave heights using satellite altimeter data. https://doi.org/10.1016/j.ocemod.2018.09.001

</details>

---


> **For more information check the BlueMath Repo:** 
>
> https://github.com/GeoOcean/BlueMath/blob/main/toolkit/waves/CalVal_Carolinas.ipynb

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import os.path as op
from numpy import atan2
import wavespectra

In [ ]:
seapoint_to_sample = 1903793
station_lat=34.200
station_lon=-76.500+360
p_data = op.join(os.getcwd())
output_folder = op.join(p_data, 'outputs/41007_spec_WHACS.nc')

In [ ]:
Hswell1 = xr.open_dataset("../inputs/WHACS/north_carolina_phs1_WHACS_Jan97.nc").phs1.sel(
    seapoint=seapoint_to_sample
)
Hs = xr.open_dataset("../inputs/WHACS/north_carolina_02_hs_WHACS.nc").hs.sel(
    seapoint=seapoint_to_sample, time=Hswell1.time
)
Dp = xr.open_dataset("../inputs/WHACS/north_carolina_02_dp_WHACS.nc").dp.sel(
    seapoint=seapoint_to_sample, time=Hswell1.time
)
fp = xr.open_dataset("../inputs/WHACS/north_carolina_02_fp_WHACS.nc").fp.sel(
    seapoint=seapoint_to_sample, time=Hswell1.time
)
# DirM = xr.open_dataset("../inputs/WHACS/north_carolina_02_dir_WHACS.nc").dir.sel(
#     seapoint=seapoint_to_sample, time=Hswell1.time
# )
Hsea = xr.open_dataset("../inputs/WHACS/north_carolina_02_phs0_WHACS.nc").phs0.sel(
    seapoint=seapoint_to_sample, time=Hswell1.time    
)

Hswell2 = xr.open_dataset("../inputs/WHACS/north_carolina_02_phs2_WHACS.nc").phs2.sel(
    seapoint=seapoint_to_sample, time=Hswell1.time
)
Hswell3 = xr.open_dataset("../inputs/WHACS/north_carolina_02_phs3_WHACS.nc").phs3.sel(
    seapoint=seapoint_to_sample, time=Hswell1.time
)
Dirsea = xr.open_dataset("../inputs/WHACS/north_carolina_02_pdp0_WHACS.nc").pdp0.sel(
    seapoint=seapoint_to_sample, time=Hswell1.time
)
Dirswell1 = xr.open_dataset("../inputs/WHACS/north_carolina_02_pdp1_WHACS.nc").pdp1.sel(
    seapoint=seapoint_to_sample, time=Hswell1.time
)
Dirswell2 = xr.open_dataset("../inputs/WHACS/north_carolina_02_pdp2_WHACS.nc").pdp2.sel(
    seapoint=seapoint_to_sample, time=Hswell1.time
)
Dirswell3 = xr.open_dataset("../inputs/WHACS/north_carolina_02_pdp3_WHACS.nc").pdp3.sel(
    seapoint=seapoint_to_sample, time= Hswell1.time   
)


In [ ]:
csiro = pd.DataFrame({
    'Hs':Hs.values,
    'Dm':Dp.values,
    'Tp':1/fp.values,
    # 'DirM':DirM.values,
    'Hsea': Hsea.values,
    'Hswell1': Hswell1.values,
    'Hswell2': Hswell2.values,
    'Hswell3': Hswell3.values,
    'Dirsea': Dirsea.values,
    'Dirswell1': Dirswell1.values,
    'Dirswell2': Dirswell2.values,
    'Dirswell3': Dirswell3.values,
}, index=pd.to_datetime(Hs.time.values))

# csiro.index = csiro.index.round('H')
lat = float(station_lat)
lon = float(station_lon)
csiro_lon, csiro_lat = lon, lat
csiro

In [ ]:
csiro.Hs['1996-11-01':'1997-03-30'].plot()

In [ ]:
buoy = (
    pd.read_pickle(
        f"inputs/buoy_data/buoy_41007_bulk_parameters.pkl"
    )
    .rename(columns={"Hs_Buoy": "Hs_CAL"})
)
buoy = buoy.dropna(subset=['Hs_CAL'])
buoy['LATITUDE']=station_lat
buoy['LONGITUDE']=station_lon
buoy

# # Remove rows where Hs_CAL = 0 and save to new file with Hs_Buoy column name
# buoy_nonzero = buoy[buoy['Hs_CAL'] >= 0.2].copy()
# buoy_nonzero = buoy_nonzero.rename(columns={"Hs_CAL": "Hs_Buoy"})
# buoy_nonzero.to_pickle("inputs/buoy_data/buoy_41025_bulk_parameters_nonzeros.pkl")


### Comparison Wave Buoy vs Raw whacs spectrum

In [ ]:
# Plot comparison between buoy validation data and CSIRO data
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.dates import DateFormatter
import matplotlib.dates as mdates

# Create time series plot
fig1, axes = plt.subplots(3, 1, figsize=(15, 12))
fig1.suptitle('Buoy vs CSIRO - Before Satellite Correction', fontsize=16, fontweight='bold')

# Find common time range
common_start = max(buoy.index.min(), csiro.index.min())
common_end = min(buoy.index.max(), csiro.index.max())

# Filter data to common time range
buoy_common = buoy[(buoy.index >= common_start) & (buoy.index <= common_end)]
csiro_common = csiro[(csiro.index >= common_start) & (csiro.index <= common_end)]

# Hs time series
axes[0].plot(buoy_common.index, buoy_common['Hs_CAL'], color='plum', label='Buoy Hs', linewidth=1.5, alpha=0.8)
axes[0].plot(csiro_common.index, csiro_common['Hs'], color='turquoise', label='CSIRO Hs', linewidth=1.5, alpha=0.8)
axes[0].set_ylabel('Hs [m]', fontsize=12)
axes[0].legend(fontsize=10)

# Tp time series
axes[1].plot(buoy_common.index, buoy_common['Tp_Buoy'], color='plum', label='Buoy Tp', linewidth=1.5, alpha=0.8)
axes[1].plot(csiro_common.index, csiro_common['Tp'], color='turquoise', label='CSIRO Tp', linewidth=1.5, alpha=0.8)
axes[1].set_ylabel('Tp [s]', fontsize=12)
axes[1].legend(fontsize=10)

# Dir time series
axes[2].plot(buoy_common.index, buoy_common['Dir_Buoy'], color='plum', label='Buoy Dir', linewidth=1.5, alpha=0.8)
axes[2].plot(csiro_common.index, csiro_common['Dm'], color='turquoise', label='CSIRO Dir', linewidth=1.5, alpha=0.8)
axes[2].set_ylabel('Direction [°]', fontsize=12)
axes[2].set_xlabel('Time', fontsize=12)
axes[2].legend(fontsize=10)
plt.tight_layout()
plt.show()

# Create scatter plots
fig2, axes = plt.subplots(1, 3, figsize=(18, 6))
fig2.suptitle('Buoy vs CSIRO - Before Satellite Correction', fontsize=16, fontweight='bold')

# Merge data on time index for scatter plots
merged_data = buoy_common.merge(csiro_common, left_index=True, right_index=True, how='inner')

# Hs scatter plot
axes[0].scatter(merged_data['Hs_CAL'], merged_data['Hs'], alpha=0.6, s=20, color='plum')
axes[0].plot([0, merged_data['Hs_CAL'].max()], [0, merged_data['Hs_CAL'].max()], 'k--', linewidth=1)
axes[0].set_xlabel('Buoy Hs [m]', fontsize=12)
axes[0].set_ylabel('CSIRO Hs [m]', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

# Tp scatter plot
axes[1].scatter(merged_data['Tp_Buoy'], merged_data['Tp'], alpha=0.6, s=20, color='plum')
axes[1].plot([0, merged_data['Tp_Buoy'].max()], [0, merged_data['Tp_Buoy'].max()], 'k--', linewidth=1)
axes[1].set_xlabel('Buoy Tp [s]', fontsize=12)
axes[1].set_ylabel('CSIRO Tp [s]', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')

# Dir scatter plot
axes[2].scatter(merged_data['Dir_Buoy'], merged_data['Dm'], alpha=0.6, s=20, color='plum')
axes[2].plot([0, 360], [0, 360], 'k--', linewidth=1)
axes[2].set_xlabel('Buoy Dir [°]', fontsize=12)
axes[2].set_ylabel('CSIRO Dir [°]', fontsize=12)
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim(0, 360)
axes[2].set_ylim(0, 360)

plt.tight_layout()
plt.show()

# Print some statistics
print("Data Statistics:")
print(f"Common time period: {common_start} to {common_end}")
print(f"Number of common data points: {len(merged_data)}")
print("\nCorrelation coefficients:")
print(f"Hs correlation: {merged_data['Hs_CAL'].corr(merged_data['Hs']):.3f}")
print(f"Tp correlation: {merged_data['Tp_Buoy'].corr(merged_data['Tp']):.3f}")
print(f"Dir correlation: {merged_data['Dir_Buoy'].corr(merged_data['DirM']):.3f}")

> ⚠️ **NOTE:** A smaller radius (0.2 degrees) was selected for this correction—compared to the 1-degree radius used in Albuquerque et al., 2018—due to abrupt depth changes around the WHACS point.


In [ ]:
# from bluemath_tk.waves.calibration import CalVal, process_imos_satellite_data


# satellite_raw = xr.open_dataset("inputs/satellite_dataset_carolinas.nc")[
#     [
#         "SWH_KU_quality_control",
#         "SWH_KA_quality_control",
#         "SWH_KU_CAL",
#         "SWH_KA_CAL",
#         "BOT_DEPTH",
#     ]
# ].to_dataframe()
# satellite_processed = process_imos_satellite_data(
#     satellite_df=satellite_raw,
#     ini_lat=station_lat-10,
#     end_lat=station_lat+10,
#     ini_lon=station_lon-10,
#     end_lon=station_lon+10,
#     depth_threshold=-100,
# )
# # satellite_processed = process_imos_satellite_data(
# #     satellite_df=satellite_raw,
# #     ini_lat=station_lat-.5,
# #     end_lat=station_lat+.5,
# #     ini_lon=station_lon-.5,
# #     end_lon=station_lon+.5,
# #     depth_threshold=-40,
# # )


# satellite_processed

In [ ]:
from bluemath_tk.waves.calibration import CalVal

In [ ]:
calval = CalVal()
calval

In [ ]:
# Build a mask that includes ALL sea and swell columns
whacs_hindcast_data = csiro
cols = ['Hsea','Dirsea'] + [c for c in whacs_hindcast_data.columns if c.lower().startswith('hswell')]
cols += [c for c in whacs_hindcast_data.columns if c.lower().startswith('dirswell')]

mask = whacs_hindcast_data[cols].apply(np.isfinite).all(axis=1)
whacs_clean = whacs_hindcast_data.loc[mask].copy()

print("NaNs left in required columns:", (~np.isfinite(whacs_clean[cols])).sum().sum())


In [ ]:
calval.fit(
    data=whacs_clean,
    data_longitude=csiro_lon,
    data_latitude=csiro_lat,
    data_to_calibrate=buoy,
    max_time_diff=1,
)

In [ ]:
calval.calibration_params

In [ ]:
calval.plot_calibration_results();

In [ ]:
corrected_csiro = calval.correct(data=csiro)
corrected_csiro

In [ ]:
sprsea = xr.open_dataset("inputs/WHACS/north_carolina_pspr0_WHACS.nc").pspr0.sel(
    seapoint=seapoint_to_sample, time= corrected_csiro.index.values
)
sprswell1 = xr.open_dataset("inputs/WHACS/north_carolina_pspr1_WHACS.nc").pspr1.sel(
    seapoint=seapoint_to_sample, time= corrected_csiro.index.values 
)
sprswell2 = xr.open_dataset("inputs/WHACS/north_carolina_pspr2_WHACS.nc").pspr2.sel(
    seapoint=seapoint_to_sample, time= corrected_csiro.index.values  
)
sprswell3 = xr.open_dataset("inputs/WHACS/north_carolina_pspr3_WHACS.nc").pspr3.sel(
    seapoint=seapoint_to_sample, time= corrected_csiro.index.values
)

tpsea = xr.open_dataset("inputs/WHACS/north_carolina_ptp0_WHACS.nc").ptp0.sel(
    seapoint=seapoint_to_sample, time= corrected_csiro.index.values
)

tpswell1 = xr.open_dataset("inputs/WHACS/north_carolina_ptp1_WHACS.nc").ptp1.sel(
    seapoint=seapoint_to_sample, time= corrected_csiro.index.values
)

tpswell2 = xr.open_dataset("inputs/WHACS/north_carolina_ptp2_WHACS.nc").ptp2.sel(
    seapoint=seapoint_to_sample, time= corrected_csiro.index.values
)

tpswell3 = xr.open_dataset("inputs/WHACS/north_carolina_ptp3_WHACS.nc").ptp3.sel(
    seapoint=seapoint_to_sample, time= corrected_csiro.index.values
)

In [ ]:
corrected_csiro['Tpsea'] = tpsea
corrected_csiro['Tpswell1'] = tpswell1
corrected_csiro['Tpswell2'] = tpswell2
corrected_csiro['Tpswell3'] = tpswell3
corrected_csiro['Sprsea'] = sprsea
corrected_csiro['Sprswell1'] = sprswell1
corrected_csiro['Sprswell2'] = sprswell2
corrected_csiro['Sprswell3'] = sprswell3


In [ ]:
corrected_csiro

In [ ]:
# freq = np.arange(0.03, 0.401, 0.01)
# dir = np.arange(0, 360, 12)
freq = np.array([0.035,      0.03848704, 0.0423215,  0.04653798, 0.05117455, 0.05627306,
 0.06187953, 0.06804458, 0.07482385, 0.08227853, 0.09047592, 0.09949002,
 0.10940219, 0.12030191, 0.13228757, 0.14546735, 0.15996023, 0.17589704,
 0.19342162, 0.21269218, 0.23388266, 0.25718434, 0.28280756, 0.31098362,
 0.34196685, 0.37603694, 0.41350142, 0.45469848, 0.5       ])
dir = np.arange(0, 360, 5)

In [ ]:
time_to_sample = slice(corrected_csiro.index.values) 
seapoint_to_sample = seapoint_to_sample

# Map the DataFrame columns to xarray DataArrays for each partition
# Assuming corrected_csiro has columns: Hsea, Hswell1, Hswell2, Hswell3
# and corresponding Tpsea, Tpswell1, etc. and Dirsea, Dirswell1, etc.

# Define mapping for each partition (0=sea, 1-3=swell)
partition_names = {
    0: 'sea',
    1: 'swell1', 
    2: 'swell2',
    3: 'swell3'
}

whacs_sample = xr.Dataset(
    {
        "hs": xr.concat(
            [
                xr.DataArray(
                    corrected_csiro[f"Hsea" if i == 0 else f"Hswell{i}"].values,
                    dims=["time"],
                    coords={"time": corrected_csiro.index}
                )
                .expand_dims({"part": [i]})
                for i in range(4)
            ],
            dim="part",
        ),
        "tp": xr.concat(
            [
                xr.DataArray(
                    corrected_csiro[f"Tpsea" if i == 0 else f"Tpswell{i}"].values,
                    dims=["time"],
                    coords={"time": corrected_csiro.index}
                ).expand_dims({"part": [i]})
                for i in range(4)
            ],
            dim="part",
        ),
        "dp": xr.concat(
            [
                xr.DataArray(
                    corrected_csiro[f"Dirsea" if i == 0 else f"Dirswell{i}"].values,
                    dims=["time"],
                    coords={"time": corrected_csiro.index}
                ).expand_dims({"part": [i]})
                for i in range(4)
            ],
            dim="part",
        ),
        "spr": xr.concat(
            [
                xr.DataArray(
                    corrected_csiro[f"Sprsea" if i == 0 else f"Sprswell{i}"].values,
                    dims=["time"],
                    coords={"time": corrected_csiro.index}
                ).expand_dims({"part": [i]})
                for i in range(4)
            ],
            dim="part",
        ),
    }
)
whacs_sample

In [ ]:
from wavespectra.construct import construct_partition
full_spec = construct_partition(
    freq_name="jonswap",
    freq_kwargs={
        "freq": freq,
        "fp": 1 / whacs_sample.tp,
        "hs": np.sqrt(whacs_sample.hs),
    },
    dir_name="cartwright",
    dir_kwargs={
        "dir": dir,
        "dm": whacs_sample.dp,
        "dspr": whacs_sample.spr,
    },
)
full_spec

In [ ]:
# Fix the subplot creation
fig, ax = plt.subplots(figsize=[20, 7])

ax.scatter(corrected_csiro.Hs, corrected_csiro.Hs_CORR, c='plum', s=3)
ax.scatter(corrected_csiro.Hs, full_spec.sum("part").spec.hs(), c='turquoise', s=1)
ax.plot([0, np.nanmax(corrected_csiro.Hs)+.2], [0, np.nanmax(corrected_csiro.Hs)+.2], '--', color='gray')
ax.set_aspect('equal')
ax.set_xlim([0, np.nanmax(corrected_csiro.Hs)+.2])
ax.set_ylim([0, np.nanmax(corrected_csiro.Hs)+.2])
ax.grid(color='coral', alpha=.2)
ax.set_xlabel('CSIRO Original Hs [m]', fontsize=15)
ax.set_ylabel('CSIRO Corrected Hs [m]', fontsize=15)

In [ ]:
# Convert full_spec time to pandas DatetimeIndex
spec_times = pd.DatetimeIndex(full_spec.time.values)

# Find intersection with buoy times
common_times = buoy.index.intersection(spec_times)
print(f"Common timestamps: {len(common_times)}")

# Filter both to common times
buoy_common = buoy.loc[common_times]
csiro_common = full_spec.sel(time=common_times)

print(f"Buoy shape: {buoy_common.shape}")
print(f"Spec shape: {csiro_common.sizes}")

In [ ]:
# Create time series plot
fig1, axes = plt.subplots(3, 1, figsize=(15, 12))
fig1.suptitle('Buoy vs CSIRO - Before Satellite Correction', fontsize=16, fontweight='bold')

# Hs time series
axes[0].plot(buoy_common.index, buoy_common['Hs_CAL'], color='plum', label='Buoy Hs', linewidth=1.5, alpha=0.8)
axes[0].plot(csiro_common.time, csiro_common.sum("part").spec.hs(), color='turquoise', label='CSIRO Hs', linewidth=1.5, alpha=0.8)
axes[0].set_ylabel('Hs [m]', fontsize=12)
axes[0].legend(fontsize=10)

# # Tp time series
# axes[1].plot(buoy_common.index, buoy_common['Tp_Buoy'], color='plum', label='Buoy Tp', linewidth=1.5, alpha=0.8)
# axes[1].plot(csiro_common.index, csiro_common['Tp'], color='turquoise', label='CSIRO Tp', linewidth=1.5, alpha=0.8)
# axes[1].set_ylabel('Tp [s]', fontsize=12)
# axes[1].legend(fontsize=10)

# # Dir time series
# axes[2].plot(buoy_common.index, buoy_common['Dir_Buoy'], color='plum', label='Buoy Dir', linewidth=1.5, alpha=0.8)
# axes[2].plot(csiro_common.index, csiro_common['Dm'], color='turquoise', label='CSIRO Dir', linewidth=1.5, alpha=0.8)
# axes[2].set_ylabel('Direction [°]', fontsize=12)
# axes[2].set_xlabel('Time', fontsize=12)
# axes[2].legend(fontsize=10)
# plt.tight_layout()
# plt.show()



In [ ]:
# Find common time range
common_start = max(buoy.index.min(), csiro.index.min())
common_end = min(buoy.index.max(), csiro.index.max())

# Filter data to common time range
buoy_common_ori = buoy[(buoy.index >= common_start) & (buoy.index <= common_end)]
csiro_common_ori = csiro[(csiro.index >= common_start) & (csiro.index <= common_end)]

# Merge data on time index for scatter plots (ADD THIS LINE)
merged_data_ori = buoy_common_ori.merge(csiro_common_ori, left_index=True, right_index=True, how='inner')

# Create scatter plots
fig2, axes = plt.subplots(1, 3, figsize=(18, 6))
fig2.suptitle('Buoy vs CSIRO - Before Satellite Correction', fontsize=16, fontweight='bold')

# Merge data on time index for scatter plots


# Hs scatter plot - USE MERGED DATA INSTEAD
axes[0].scatter(merged_data_ori['Hs_CAL'], merged_data_ori['Hs'], alpha=0.6, s=20, color='turquoise')
axes[0].scatter(buoy_common['Hs_CAL'], csiro_common.sum("part").spec.hs(), alpha=0.6, s=20, color='plum')

axes[0].plot([0, merged_data_ori['Hs_CAL'].max()], [0, merged_data_ori['Hs_CAL'].max()], 'k--', linewidth=1)
axes[0].set_xlabel('Buoy Hs [m]', fontsize=12)
axes[0].set_ylabel('CSIRO Hs [m]', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

# Tp scatter plot
axes[1].scatter(merged_data['Tp_Buoy'], merged_data['Tp'], alpha=0.6, s=20, color='plum')
axes[1].plot([0, merged_data['Tp_Buoy'].max()], [0, merged_data['Tp_Buoy'].max()], 'k--', linewidth=1)
axes[1].set_xlabel('Buoy Tp [s]', fontsize=12)
axes[1].set_ylabel('CSIRO Tp [s]', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')

# Dir scatter plot
axes[2].scatter(merged_data['Dir_Buoy'], merged_data['Dm'], alpha=0.6, s=20, color='plum')
axes[2].plot([0, 360], [0, 360], 'k--', linewidth=1)
axes[2].set_xlabel('Buoy Dir [°]', fontsize=12)
axes[2].set_ylabel('CSIRO Dir [°]', fontsize=12)
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim(0, 360)
axes[2].set_ylim(0, 360)

plt.tight_layout()
plt.show()

In [ ]:
full_spec.sum("part").to_netcdf("outputs/WHACS_spectra_5D/41007_spec_WHACS_buoy_correted_5D.nc")